# 05 Multivariate EDA

This notebook develops deeper operational and executive-level insights from the completed analytical taxi dataset. It examines how time, geography, payment behavior, distance, duration, revenue, and congestion fees interact.

The work is descriptive business intelligence only. No predictive modeling or machine learning is performed.

## TOR Focus

Section 6.3 requires multivariate EDA across combinations such as time x geography x demand, geography x payment type x revenue, distance x duration x total amount, month x hour x borough or zone, and pre/post January 5, 2025 patterns where `cbd_congestion_fee` is relevant. Section 5.2 requires concise executive insights, charts, business implications, and caveats suitable for a non-technical report.

In [ ]:
from pathlib import Path
import os
import tempfile

MPLCONFIG_DIR = Path(tempfile.gettempdir()) / "data_vis_project_matplotlib"
MPLCONFIG_DIR.mkdir(exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIG_DIR)

import numpy as np
import pandas as pd
import matplotlib

if "get_ipython" not in globals():
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 140)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (11, 5.5),
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.titleweight": "bold",
})

PRIMARY = "#1f77b4"
SECONDARY = "#4c78a8"
ACCENT = "#f58518"
GREEN = "#54a24b"
RED = "#e45756"
NEUTRAL = "#6b7280"

## Load Analytical Dataset and Review Multivariate Candidates

The notebook uses the completed analytical parquet. Time-based analysis is restricted to Q1 2025 so a small number of anomalous date records do not affect executive trend charts.

In [ ]:
PROJECT_ROOT = Path.cwd()
ANALYTICAL_PATH = PROJECT_ROOT / "Data" / "processed" / "analytical" / "analytical.parquet"
if not ANALYTICAL_PATH.exists():
    raise FileNotFoundError(f"Analytical dataset not found: {ANALYTICAL_PATH}")

COLUMNS = [
    "pickup_date", "pickup_month", "pickup_hour", "pickup_day", "is_weekend",
    "pickup_borough", "dropoff_borough", "pickup_zone", "dropoff_zone",
    "payment_type", "trip_distance", "trip_duration_min", "avg_speed_mph",
    "fare_amount", "total_amount", "tip_percent", "cbd_period", "cbd_congestion_fee",
]

eda = pd.read_parquet(ANALYTICAL_PATH, columns=COLUMNS)
eda["pickup_date"] = pd.to_datetime(eda["pickup_date"], errors="coerce")
eda["pickup_month"] = eda["pickup_month"].astype("string")

q1_mask = eda["pickup_date"].between(pd.Timestamp("2025-01-01"), pd.Timestamp("2025-03-31"))
analysis = eda.loc[q1_mask].copy()
out_of_period_rows = int((~q1_mask).sum())

print(f"Loaded: {ANALYTICAL_PATH}")
print(f"Rows loaded: {len(eda):,}")
print(f"Rows used for Q1 multivariate EDA: {len(analysis):,}")
print(f"Rows outside Q1 excluded from time-based charts: {out_of_period_rows:,}")
analysis.head()

In [ ]:
feature_review = pd.DataFrame({
    "dimension_group": [
        "time", "time", "time", "time", "time",
        "geography", "geography", "geography", "geography",
        "payment", "distance", "duration", "efficiency", "revenue", "revenue", "payment", "policy", "policy",
    ],
    "column": COLUMNS,
    "dtype": [str(analysis[column].dtype) for column in COLUMNS],
    "missing_rows": [int(analysis[column].isna().sum()) for column in COLUMNS],
    "missing_pct": [analysis[column].isna().mean() * 100 for column in COLUMNS],
})
feature_review

**Strong candidate dimensions:** time (`pickup_month`, `pickup_hour`, `pickup_day`), geography (`pickup_borough`, `pickup_zone`, `dropoff_borough`, `dropoff_zone`), payment behavior (`payment_type`, `tip_percent`), trip mechanics (`trip_distance`, `trip_duration_min`, `avg_speed_mph`), revenue (`fare_amount`, `total_amount`), and CBD policy fields (`cbd_period`, `cbd_congestion_fee`).

## Helper Functions and Analysis Views

In [ ]:
WEEKDAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
MONTH_ORDER = ["2025-01", "2025-02", "2025-03"]
PAYMENT_LABELS = {
    0: "0 - Flex/missing",
    1: "1 - Credit card",
    2: "2 - Cash",
    3: "3 - No charge",
    4: "4 - Dispute",
    5: "5 - Unknown",
    6: "6 - Voided",
}

analysis["payment_label"] = analysis["payment_type"].map(PAYMENT_LABELS).fillna("Other")
analysis["day_type"] = np.where(analysis["is_weekend"].fillna(False), "Weekend", "Weekday")
analysis["positive_cbd_fee"] = analysis["cbd_congestion_fee"].gt(0)

distance_p99 = analysis["trip_distance"].quantile(0.99)
duration_p99 = analysis["trip_duration_min"].quantile(0.99)
total_p99 = analysis["total_amount"].quantile(0.99)
fare_p99 = analysis["fare_amount"].quantile(0.99)

ops = analysis[
    analysis["trip_distance"].gt(0)
    & analysis["trip_distance"].le(distance_p99)
    & analysis["trip_duration_min"].gt(0)
    & analysis["trip_duration_min"].le(duration_p99)
    & analysis["total_amount"].gt(0)
    & analysis["total_amount"].le(total_p99)
    & analysis["fare_amount"].gt(0)
    & analysis["fare_amount"].le(fare_p99)
].copy()

print(f"Operational metric view rows: {len(ops):,}")
print(f"Distance p99: {distance_p99:,.2f} miles | Duration p99: {duration_p99:,.2f} min | Total p99: ${total_p99:,.2f}")

In [ ]:
def save_table(df, name):
    """Keep table creation explicit without writing files from this notebook."""
    return df


def save_fig(name, presentation_worthy=False):
    """Display charts in the notebook without writing image files."""
    plt.tight_layout()
    plt.show()
    return None


def sample_rows(df, n=120_000, random_state=42):
    return df if len(df) <= n else df.sample(n=n, random_state=random_state)


def heatmap_from_pivot(pivot, title, xlabel, ylabel, name, cmap="Blues", presentation_worthy=False):
    fig, ax = plt.subplots(figsize=(12, 6.5))
    sns.heatmap(pivot, cmap=cmap, annot=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    save_fig(name, presentation_worthy=presentation_worthy)

# 1. Time x Geography x Demand

In [ ]:
top_boroughs = analysis["pickup_borough"].value_counts().head(5).index.tolist()
top_zones = analysis["pickup_zone"].value_counts().head(12).index.tolist()

borough_hour = analysis[analysis["pickup_borough"].isin(top_boroughs)].pivot_table(
    index="pickup_borough", columns="pickup_hour", values="total_amount", aggfunc="size", fill_value=0
)
borough_weekday = analysis[analysis["pickup_borough"].isin(top_boroughs)].pivot_table(
    index="pickup_borough", columns="pickup_day", values="total_amount", aggfunc="size", fill_value=0
).reindex(columns=WEEKDAY_ORDER)
zone_hour = analysis[analysis["pickup_zone"].isin(top_zones)].pivot_table(
    index="pickup_zone", columns="pickup_hour", values="total_amount", aggfunc="size", fill_value=0
).loc[top_zones]

save_table(borough_hour, "time_geography_borough_hour_trip_counts")
save_table(borough_weekday, "time_geography_borough_weekday_trip_counts")
save_table(zone_hour, "time_geography_top_zone_hour_trip_counts")
borough_hour

In [ ]:
heatmap_from_pivot(borough_hour / 1_000, "Pickup Demand by Borough and Hour (Thousands of Trips)", "Pickup hour", "Pickup borough", "presentation_time_geography_borough_hour_heatmap", presentation_worthy=True)
heatmap_from_pivot(borough_weekday / 1_000, "Pickup Demand by Borough and Weekday (Thousands of Trips)", "Pickup day", "Pickup borough", "time_geography_borough_weekday_heatmap", cmap="Greens")
heatmap_from_pivot(zone_hour / 1_000, "Top Pickup Zone Demand by Hour (Thousands of Trips)", "Pickup hour", "Pickup zone", "presentation_time_geography_zone_hour_heatmap", cmap="YlGnBu", presentation_worthy=True)

borough_hour_long = analysis[analysis["pickup_borough"].isin(top_boroughs)].groupby(["pickup_hour", "pickup_borough"]).size().reset_index(name="trip_count")
fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=borough_hour_long, x="pickup_hour", y="trip_count", hue="pickup_borough", marker="o", linewidth=2.2, ax=ax)
ax.set_title("Hourly Demand Pattern by Pickup Borough")
ax.set_xlabel("Pickup hour")
ax.set_ylabel("Trip count")
ax.set_xticks(range(24))
ax.legend(frameon=False, title="Pickup borough")
save_fig("time_geography_borough_hour_lines")

**Observation:** Manhattan dominates demand across almost all hours, with the strongest concentration in late afternoon and evening.

**Interpretation:** Time and place interact strongly: the same hour has different operational meaning depending on borough and zone.

**Business implication:** Dispatch and fleet positioning should be managed by hour and geography together, especially for Manhattan evening peaks and airport zone coverage.

# 2. Geography x Payment Type x Revenue

In [ ]:
main_payments = ["1 - Credit card", "2 - Cash", "0 - Flex/missing", "4 - Dispute", "3 - No charge"]
geo_payment = analysis[analysis["payment_label"].isin(main_payments)].groupby(["pickup_borough", "payment_label"]).agg(
    trips=("total_amount", "size"), recorded_revenue=("total_amount", "sum")
).reset_index()
geo_payment["recorded_revenue_m"] = geo_payment["recorded_revenue"] / 1_000_000

geo_tip_payment = ops[ops["payment_label"].isin(main_payments)].groupby(["pickup_borough", "payment_label"]).agg(
    trips=("total_amount", "size"), avg_tip_percent=("tip_percent", "mean"), avg_total_amount=("total_amount", "mean")
).reset_index()

revenue_pivot = geo_payment.pivot_table(index="pickup_borough", columns="payment_label", values="recorded_revenue_m", aggfunc="sum", fill_value=0)
tip_pivot = geo_tip_payment.pivot_table(index="pickup_borough", columns="payment_label", values="avg_tip_percent", aggfunc="mean")

save_table(geo_payment, "geography_payment_revenue_summary")
save_table(geo_tip_payment, "geography_payment_tip_summary")
geo_payment.sort_values("recorded_revenue", ascending=False).head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=geo_payment[geo_payment["pickup_borough"].isin(top_boroughs)], x="pickup_borough", y="recorded_revenue_m", hue="payment_label", palette="Blues", ax=ax)
ax.set_title("Recorded Revenue by Pickup Borough and Payment Type")
ax.set_xlabel("Pickup borough")
ax.set_ylabel("Recorded revenue ($ millions)")
ax.tick_params(axis="x", rotation=30)
ax.legend(frameon=False)
save_fig("presentation_geography_payment_revenue_grouped_bar", presentation_worthy=True)

heatmap_from_pivot(revenue_pivot.loc[revenue_pivot.sum(axis=1).sort_values(ascending=False).index], "Revenue by Borough and Payment Type ($ Millions)", "Payment type", "Pickup borough", "geography_payment_revenue_heatmap", cmap="Greens")
heatmap_from_pivot(tip_pivot.loc[revenue_pivot.sum(axis=1).sort_values(ascending=False).index], "Average Tip Percent by Borough and Payment Type", "Payment type", "Pickup borough", "geography_payment_tip_heatmap", cmap="YlOrBr")

**Observation:** Credit card revenue dominates in Manhattan and other high-volume areas. Recorded tip percentages are materially higher for card payments than cash across boroughs.

**Interpretation:** Payment behavior is both a customer behavior signal and a data-capture issue.

**Business implication:** Executive tip and driver-earning discussions should separate card, cash, and incomplete payment metadata.

# 3. Distance x Duration x Total Amount

In [ ]:
scatter_view = ops[["trip_distance", "trip_duration_min", "avg_speed_mph", "total_amount"]].dropna()
scatter_sample = sample_rows(scatter_view, n=140_000)
corr = scatter_view[["trip_distance", "trip_duration_min", "avg_speed_mph", "total_amount"]].corr()
save_table(corr, "distance_duration_total_amount_correlation")

fig, ax = plt.subplots(figsize=(11, 6.5))
points = ax.scatter(scatter_sample["trip_distance"], scatter_sample["trip_duration_min"], c=scatter_sample["total_amount"], s=np.clip(scatter_sample["total_amount"], 5, 80), cmap="viridis", alpha=0.18)
ax.set_title("Distance x Duration x Total Amount (Sampled)")
ax.set_xlabel("Trip distance (miles)")
ax.set_ylabel("Trip duration (minutes)")
cbar = plt.colorbar(points, ax=ax)
cbar.set_label("Total amount ($)")
save_fig("presentation_distance_duration_total_bubble", presentation_worthy=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation: Distance, Duration, Speed, Total Amount")
save_fig("distance_duration_total_correlation_heatmap")

In [ ]:
binned_efficiency = ops.assign(
    distance_band=pd.cut(ops["trip_distance"], bins=[0, 1, 2, 5, 10, 20], right=False),
    duration_band=pd.cut(ops["trip_duration_min"], bins=[0, 5, 10, 20, 40, 80], right=False),
).pivot_table(index="duration_band", columns="distance_band", values="total_amount", aggfunc="median", observed=True)
save_table(binned_efficiency, "distance_duration_median_total_amount_matrix")
heatmap_from_pivot(binned_efficiency, "Median Total Amount by Distance and Duration Band", "Distance band (miles)", "Duration band (minutes)", "distance_duration_total_amount_heatmap", cmap="YlGnBu")

**Observation:** Total amount rises strongly with distance and also increases with duration, but high-duration short-distance trips reveal inefficient urban conditions.

**Interpretation:** Congestion and slow movement can raise time cost without producing proportional distance.

**Business implication:** Efficiency reporting should monitor distance, duration, speed, and payment together.

# 4. Month x Hour x Borough

In [ ]:
month_hour_borough = analysis[analysis["pickup_borough"].isin(top_boroughs)].groupby(["pickup_month", "pickup_borough", "pickup_hour"]).size().reset_index(name="trip_count")
save_table(month_hour_borough, "month_hour_borough_trip_counts")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), sharey=True)
for ax, month in zip(axes, MONTH_ORDER):
    pivot = month_hour_borough[month_hour_borough["pickup_month"].eq(month)].pivot_table(
        index="pickup_borough", columns="pickup_hour", values="trip_count", aggfunc="sum", fill_value=0
    ).reindex(index=top_boroughs)
    sns.heatmap(pivot / 1_000, cmap="Blues", ax=ax, cbar=month == MONTH_ORDER[-1])
    ax.set_title(f"{month}: Borough x Hour Demand")
    ax.set_xlabel("Pickup hour")
    ax.set_ylabel("Pickup borough" if month == MONTH_ORDER[0] else "")
save_fig("presentation_month_hour_borough_faceted_heatmaps", presentation_worthy=True)

month_hour_total = analysis.groupby(["pickup_month", "pickup_hour"]).size().reset_index(name="trip_count")
fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=month_hour_total, x="pickup_hour", y="trip_count", hue="pickup_month", marker="o", linewidth=2.2, ax=ax)
ax.set_title("Hourly Demand Pattern by Month")
ax.set_xlabel("Pickup hour")
ax.set_ylabel("Trip count")
ax.set_xticks(range(24))
ax.legend(frameon=False, title="Month")
save_fig("month_hour_total_demand_lines")

**Observation:** The evening peak persists across months, while March carries visibly higher volume than January and February in many hour bands.

**Interpretation:** The operating rhythm is stable, but demand scale increases through the quarter.

**Business implication:** Planning should combine stable hourly staffing patterns with month-level volume adjustments.

# 5. Congestion Fee Analysis: Pre/Post January 5, 2025

In [ ]:
cbd = analysis.copy()
cbd["cbd_period_label"] = cbd["cbd_period"].map({"pre_2025_01_05": "Pre Jan 5", "post_2025_01_05": "Post Jan 5"}).fillna("Unknown")
cbd_period_summary = cbd.groupby("cbd_period_label").agg(
    trips=("total_amount", "size"),
    active_days=("pickup_date", "nunique"),
    recorded_revenue=("total_amount", "sum"),
    avg_total_amount=("total_amount", "mean"),
    avg_fare_amount=("fare_amount", "mean"),
    avg_trip_distance=("trip_distance", "mean"),
    avg_trip_duration=("trip_duration_min", "mean"),
    avg_cbd_fee=("cbd_congestion_fee", "mean"),
    positive_cbd_fee_share=("positive_cbd_fee", "mean"),
).reindex(["Pre Jan 5", "Post Jan 5"])
cbd_period_summary["trips_per_day"] = cbd_period_summary["trips"] / cbd_period_summary["active_days"]
cbd_period_summary["revenue_per_day"] = cbd_period_summary["recorded_revenue"] / cbd_period_summary["active_days"]

cbd_daily = cbd.groupby(["pickup_date", "cbd_period_label"]).agg(
    trips=("total_amount", "size"),
    recorded_revenue=("total_amount", "sum"),
    avg_total_amount=("total_amount", "mean"),
    avg_cbd_fee=("cbd_congestion_fee", "mean"),
    positive_cbd_fee_share=("positive_cbd_fee", "mean"),
).reset_index()
cbd_borough = cbd.groupby(["cbd_period_label", "pickup_borough"]).agg(
    trips=("total_amount", "size"),
    avg_cbd_fee=("cbd_congestion_fee", "mean"),
    positive_cbd_fee_share=("positive_cbd_fee", "mean"),
    avg_total_amount=("total_amount", "mean"),
    recorded_revenue=("total_amount", "sum"),
).reset_index()

save_table(cbd_period_summary, "cbd_pre_post_kpi_summary")
save_table(cbd_daily, "cbd_daily_usage_trends")
save_table(cbd_borough, "cbd_borough_pre_post_summary")
cbd_period_summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
plot_df = cbd_period_summary.reset_index()
sns.barplot(data=plot_df, x="cbd_period_label", y="trips_per_day", color=PRIMARY, ax=axes[0])
axes[0].set_title("Trips per Day")
axes[0].set_xlabel("CBD period")
axes[0].set_ylabel("Trips per day")
sns.barplot(data=plot_df, x="cbd_period_label", y="avg_total_amount", color=GREEN, ax=axes[1])
axes[1].set_title("Average Total Amount")
axes[1].set_xlabel("CBD period")
axes[1].set_ylabel("Average total amount ($)")
sns.barplot(data=plot_df, x="cbd_period_label", y="positive_cbd_fee_share", color=ACCENT, ax=axes[2])
axes[2].set_title("Trips with Positive CBD Fee")
axes[2].set_xlabel("CBD period")
axes[2].set_ylabel("Share of trips")
axes[2].set_ylim(0, 1)
save_fig("presentation_cbd_pre_post_kpi_comparison", presentation_worthy=True)

fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=cbd_daily, x="pickup_date", y="positive_cbd_fee_share", color=ACCENT, linewidth=2.2, ax=ax)
ax.axvline(pd.Timestamp("2025-01-05"), color=RED, linestyle="--", linewidth=2, label="Jan 5, 2025")
ax.set_title("Daily Share of Trips with Positive CBD Congestion Fee")
ax.set_xlabel("Pickup date")
ax.set_ylabel("Share with positive CBD fee")
ax.legend(frameon=False)
save_fig("presentation_cbd_daily_fee_share_trend", presentation_worthy=True)

cbd_borough_pivot = cbd_borough[cbd_borough["pickup_borough"].isin(top_boroughs)].pivot_table(
    index="pickup_borough", columns="cbd_period_label", values="positive_cbd_fee_share", aggfunc="mean"
).reindex(index=top_boroughs)
heatmap_from_pivot(cbd_borough_pivot, "Positive CBD Fee Share by Borough Before and After Jan 5", "CBD period", "Pickup borough", "cbd_borough_pre_post_heatmap", cmap="YlOrRd")

**Observation:** After January 5, positive CBD fee usage becomes a major part of trip records, especially in Manhattan-linked activity.

**Interpretation:** The CBD fee is visible in customer payment behavior and should be treated as a structural pricing change.

**Business implication:** Executive reporting should isolate pre/post January 5 comparisons when discussing Q1 revenue and customer cost.

# 6. Revenue Concentration Analysis

In [ ]:
zone_revenue = analysis.groupby("pickup_zone").agg(
    trips=("total_amount", "size"),
    recorded_revenue=("total_amount", "sum"),
    avg_total_amount=("total_amount", "mean"),
).sort_values("recorded_revenue", ascending=False)
zone_revenue["revenue_share"] = zone_revenue["recorded_revenue"] / zone_revenue["recorded_revenue"].sum()
zone_revenue["cumulative_revenue_share"] = zone_revenue["revenue_share"].cumsum()
zone_revenue["rank"] = np.arange(1, len(zone_revenue) + 1)
top_operational_combos = analysis.groupby(["pickup_borough", "pickup_zone", "pickup_hour"]).agg(
    trips=("total_amount", "size"),
    recorded_revenue=("total_amount", "sum"),
    avg_total_amount=("total_amount", "mean"),
).sort_values("recorded_revenue", ascending=False).head(25)
hour_borough_revenue = analysis.pivot_table(index="pickup_borough", columns="pickup_hour", values="total_amount", aggfunc="sum", fill_value=0).loc[top_boroughs]

save_table(zone_revenue.head(50), "revenue_concentration_top_50_zones")
save_table(top_operational_combos, "revenue_concentration_top_operational_combinations")
save_table(hour_borough_revenue, "revenue_concentration_borough_hour_matrix")
zone_revenue.head(15)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
pareto = zone_revenue.head(30).copy()
ax1.bar(pareto["rank"], pareto["recorded_revenue"] / 1_000_000, color=GREEN, alpha=0.8)
ax1.set_xlabel("Pickup zone revenue rank")
ax1.set_ylabel("Recorded revenue ($ millions)")
ax1.set_title("Revenue Concentration by Pickup Zone: Pareto View")
ax2 = ax1.twinx()
ax2.plot(pareto["rank"], pareto["cumulative_revenue_share"], color=ACCENT, marker="o", linewidth=2.3)
ax2.set_ylabel("Cumulative revenue share")
save_fig("presentation_revenue_concentration_zone_pareto", presentation_worthy=True)
heatmap_from_pivot(hour_borough_revenue / 1_000_000, "Recorded Revenue by Borough and Hour ($ Millions)", "Pickup hour", "Pickup borough", "revenue_concentration_borough_hour_heatmap", cmap="Greens")

**Observation:** A limited set of pickup zones contributes a large share of recorded revenue. Airport zones and central Manhattan zones appear repeatedly among top revenue contributors.

**Interpretation:** Revenue concentration is driven by both high trip volume and high-value trip types.

**Business implication:** Executive strategy should prioritize high-revenue zones and time windows for service reliability and dashboard monitoring.

# 7. Efficiency and Outlier Analysis

In [ ]:
short_long = ops[(ops["trip_distance"].le(2)) & (ops["trip_duration_min"].ge(30))].copy()
high_fare_low_distance = ops[(ops["trip_distance"].le(2)) & (ops["fare_amount"].ge(50))].copy()
low_speed_high_total = ops[(ops["avg_speed_mph"].le(5)) & (ops["total_amount"].ge(50))].copy()
long_trips = analysis[analysis["trip_distance"].gt(distance_p99)].copy()
expensive_trips = analysis[analysis["total_amount"].gt(total_p99)].copy()

outlier_summary = pd.DataFrame({
    "flag": [
        "Short distance + long duration",
        "High fare + low distance",
        "Low speed + high total amount",
        "Distance above p99",
        "Total amount above p99",
    ],
    "definition": [
        "trip_distance <= 2 and trip_duration_min >= 30",
        "trip_distance <= 2 and fare_amount >= 50",
        "avg_speed_mph <= 5 and total_amount >= 50",
        f"trip_distance > {distance_p99:.2f}",
        f"total_amount > {total_p99:.2f}",
    ],
    "rows": [len(short_long), len(high_fare_low_distance), len(low_speed_high_total), len(long_trips), len(expensive_trips)],
})
outlier_summary["share_of_q1_rows"] = outlier_summary["rows"] / len(analysis)

outlier_geo = pd.concat([
    short_long.assign(flag="Short distance + long duration"),
    high_fare_low_distance.assign(flag="High fare + low distance"),
    low_speed_high_total.assign(flag="Low speed + high total amount"),
]).groupby(["flag", "pickup_borough"]).agg(
    rows=("total_amount", "size"),
    avg_total_amount=("total_amount", "mean"),
    avg_duration=("trip_duration_min", "mean"),
    avg_distance=("trip_distance", "mean"),
).reset_index()
save_table(outlier_summary, "efficiency_outlier_flag_summary")
save_table(outlier_geo, "efficiency_outlier_geography_summary")
outlier_summary

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.8))
sns.barplot(data=outlier_summary, x="flag", y="rows", color=RED, ax=ax)
ax.set_title("Efficiency and Suspicious Combination Flags")
ax.set_xlabel("Outlier flag")
ax.set_ylabel("Row count")
ax.tick_params(axis="x", rotation=35)
save_fig("presentation_efficiency_outlier_flags", presentation_worthy=True)

outlier_pivot = outlier_geo.pivot_table(index="flag", columns="pickup_borough", values="rows", aggfunc="sum", fill_value=0)
heatmap_from_pivot(outlier_pivot, "Outlier Flags by Pickup Borough", "Pickup borough", "Outlier flag", "efficiency_outlier_borough_heatmap", cmap="Reds")

**Observation:** The data contains meaningful counts of short-distance long-duration trips, high-fare low-distance trips, low-speed high-total trips, and extreme distance/payment records.

**Interpretation:** Some flags likely reflect real congestion or exceptional service cases, while others may indicate corrections, meter issues, or remaining data-quality concerns.

**Business implication:** Executives should treat outlier-driven averages carefully and separate normal operating insights from suspicious or exceptional records.

# Top Multivariate Business Insights

1. Demand is strongest when time and geography align: Manhattan and central zones peak most clearly in late afternoon and evening.
2. Airport and Queens-linked activity behaves differently from Manhattan core trips: lower volume but higher average revenue and trip value.
3. Payment type changes the revenue story. Card payments dominate recorded tip behavior, while cash and incomplete payment metadata should not be blended into one tipping conclusion.
4. Distance, duration, and total amount are strongly connected, but congestion creates inefficient short-distance, high-duration trips.
5. CBD congestion fee behavior changes sharply after January 5, making pre/post analysis mandatory for any Q1 revenue interpretation.
6. Revenue is concentrated in a relatively small set of zones and hour-zone combinations, which are presentation-worthy targets for executive operations planning.

# Strongest Operational Patterns

- **Time x geography x demand:** evening Manhattan demand is the clearest operational concentration.
- **Geography x revenue:** Manhattan generates the largest total revenue; airport zones generate high-value trips.
- **Payment x tip behavior:** card transactions carry the strongest recorded tip signal.
- **Distance x duration x amount:** longer trips generally produce higher totals, while slow short trips reveal congestion inefficiency.
- **CBD pre/post:** the January 5 policy marker materially changes fee incidence and should be isolated in reporting.

# Findings Executives Should Watch Immediately

- Evening Manhattan and major transport-zone demand should drive staffing and fleet-positioning decisions.
- Airport zones deserve a separate revenue strategy because they combine high average fare with strategically important demand.
- CBD fee effects should be clearly explained in every Q1 financial chart that spans January 5.
- Tipping analysis should not combine card and cash records without caveats.
- Outlier combinations should be monitored because they can distort average fare, speed, and duration metrics.